# ToT Fixed -- Depth=3 Ablation (Colab)

**Goal:** run the ToT Fixed implementation with `depth=3` and compare against
the canonical `depth=6` results.

## One-time setup
1. **API Key:** Secrets panel -> add `GROQ_API_KEY` -> enable 'Notebook access'
2. **Data:** upload `gsm8k_test.json` to `MyDrive/NLP_ToT_Depth3/data/`

## For each part
In the configuration cell, set `PART` to 1, 2, or 3 -> **Runtime -> Run all**


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE    = '/content/drive/MyDrive/NLP_ToT_Depth3'
DRIVE_DATA    = os.path.join(DRIVE_BASE, 'data')
DRIVE_RESULTS = os.path.join(DRIVE_BASE, 'results')

os.makedirs(DRIVE_DATA,    exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print('Drive mounted.')
print('Data dir   :', DRIVE_DATA)
print('Results dir:', DRIVE_RESULTS)

In [ ]:
# 2. Install Groq SDK
!pip install groq -q
print('groq package ready.')

In [ ]:
# 3. Load API key
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY').strip()
if not GROQ_API_KEY:
    raise ValueError('GROQ_API_KEY not found. Add it from the Secrets panel.')
print('API key loaded.')

In [ ]:
# Configuration -- only change PART for each run
PART      = 1   # set to 1, 2, or 3

DEPTH     = 3   # ablation parameter: keep fixed
BRANCHING = 2
MODEL     = 'llama-3.1-8b-instant'

CHECKPOINT_FILE = os.path.join(DRIVE_RESULTS, 'tot_fixed_depth3_results.json')
DATA_FILE       = os.path.join(DRIVE_DATA,    'gsm8k_test.json')

PART_SLICES = {1: (0, 34), 2: (34, 67), 3: (67, 100)}
start, end  = PART_SLICES[PART]

print(f'Part {PART}: problems {start+1}-{end}')
print(f'depth={DEPTH}, branching={BRANCHING}, model={MODEL}')
print(f'Checkpoint: {CHECKPOINT_FILE}')

In [ ]:
# 4. Data file check
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Data file not found: {DATA_FILE}\n'
        'Please upload gsm8k_test.json to Drive/NLP_ToT_Depth3/data/.'
    )
print('Data file found.')

In [ ]:
# 5. LLM client (Groq)
import time
from groq import Groq

client = Groq(api_key=GROQ_API_KEY, timeout=120.0)

# Connectivity smoke test
try:
    _test = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'Say: hello'}],
        max_tokens=5,
    )
    print('Connectivity test:', _test.choices[0].message.content)
except Exception as e:
    print(f'CONNECTIVITY TEST FAILED: {type(e).__name__}: {e}')
    raise

def chat(prompt, temperature=0.0, max_tokens=1024):
    for attempt in range(8):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            time.sleep(8)
            return response.choices[0].message.content.strip()
        except Exception as e:
            err_type = type(e).__name__
            print(f'\n  Error [{attempt+1}/8] ({err_type}): {str(e)[:150]}')
            wait = 60 * (attempt + 1) if 'RateLimit' in err_type or '429' in str(e) else 15
            if attempt < 7:
                print(f'  waiting {wait}s...')
                time.sleep(wait)
    raise RuntimeError('Failed after 8 attempts.')

print('LLM client ready.')

In [ ]:
# 6. Load data (seed=42 -> identical 100 problems to the depth=6 run)
import json
import random

with open(DATA_FILE, encoding='utf-8') as f:
    all_data = json.load(f)

random.seed(42)
all_100      = random.sample(all_data, 100)
part_samples = all_100[start:end]

print(f'Total test set: {len(all_data)} problems')
print(f'This part     : {len(part_samples)} problems (index {start}-{end-1})')

In [ ]:
# 7. ToT prompts and solver
import re

THOUGHT_PROMPT = """Solve this math problem step by step. Generate the next single reasoning step only.

Problem: {problem}
Steps so far: {steps}

Next step:"""

EVALUATE_PROMPT = """Problem: {problem}
Reasoning so far: {steps}
Candidate next step: {candidate}

Is this step leading toward the correct solution? Reply with one word: sure, maybe, or impossible."""

EXTRACT_PROMPT = """Problem: {problem}
Reasoning:
{reasoning}

Based on the reasoning above, what is the final numeric answer?
Write only the number on the last line in this exact format:
#### <number>"""


def extract_answer(text):
    m = re.search(r'####\s*([\d,\.]+)', text)
    if m:
        return float(m.group(1).replace(',', ''))
    nums = re.findall(r'[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[-1]) if nums else None


def gold_answer(text):
    m = re.search(r'####\s*([\d,\.]+)', text)
    return float(m.group(1).replace(',', '')) if m else None


def solve(problem):
    beam = ['']
    for _ in range(DEPTH):
        candidates = []
        for path in beam:
            for _ in range(BRANCHING):
                thought = chat(THOUGHT_PROMPT.format(problem=problem, steps=path), temperature=0.7)
                score   = chat(EVALUATE_PROMPT.format(problem=problem, steps=path, candidate=thought), temperature=0.0)
                candidates.append((path + '\n' + thought, score))
        order = {'sure': 0, 'maybe': 1, 'impossible': 2}
        candidates.sort(key=lambda x: order.get(x[1].strip().lower().split()[0] if x[1].strip() else 'maybe', 3))
        beam = [c[0] for c in candidates[:BRANCHING]]

    best_path = beam[0]
    final_resp = chat(EXTRACT_PROMPT.format(problem=problem, reasoning=best_path), temperature=0.0)
    predicted = extract_answer(final_resp)
    if predicted is None:
        predicted = extract_answer(best_path)

    return {'reasoning': best_path, 'final_response': final_resp, 'predicted': predicted}

print('Prompts and solver ready.')

In [ ]:
# 8. Run -- checkpoint written to Drive after every problem

results        = []
done_questions = set()

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding='utf-8') as f:
        ckpt = json.load(f)
    results        = ckpt.get('results', [])
    done_questions = {r['question'] for r in results}
    print(f'Checkpoint found: {len(results)} problems already done.')
else:
    print('No checkpoint; starting from scratch.')

for i, item in enumerate(part_samples, start + 1):
    if item['question'] in done_questions:
        print(f'[{i}/100] skipping (already done)', end='\r', flush=True)
        continue

    print(f'[{i}/100] solving...', end='\r', flush=True)

    out  = solve(item['question'])
    gold = gold_answer(item['answer'])

    results.append({
        'question':       item['question'],
        'gold':           gold,
        'predicted':      out['predicted'],
        'correct':        out['predicted'] == gold,
        'reasoning':      out['reasoning'],
        'final_response': out['final_response'],
    })

    correct_so_far = sum(r['correct'] for r in results)
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'strategy': f'tot_fixed_depth{DEPTH}_branch{BRANCHING}',
            'model':    MODEL,
            'n_done':   len(results),
            'accuracy': round(correct_so_far / len(results) * 100, 1),
            'results':  results,
        }, f, indent=2, ensure_ascii=False)

correct = sum(r['correct'] for r in results)
n       = len(results)
print(f'\nPart {PART} finished: {correct}/{n} = {correct/n*100:.1f}%')